# LLM-Based Feature Selection for ICU Delirium Prediction

Candidate clinical features for ICU delirium prediction are prioritized by three large
language models using four query strategies.

## Methods

| Method | Query | Output |
|---|---|---|
| **LLM-Rank** | Present the feature list and request an importance-ordered ranking | Rank |
| **LLM-Score** | Present one feature at a time and request an importance score in [0, 1] | Score |
| **LLM-Seq** | Greedily select the next best feature given those already selected | Selection order |
| **LMprior** | Compute `log P(Y) − log P(N)` from token-level log-probabilities | Score |

## Models

| Vendor | Model ID | Released | Reasoning configuration |
|---|---|---|---|
| OpenAI | `gpt-4.1-2025-04-14` | 2025-04-14 | Non-reasoning model (no such parameter) |
| Google | `gemini-2.5-pro` | 2025-06-17 | Cannot be disabled — `thinkingBudget = 128` |
| Anthropic | `claude-sonnet-4-5-20250929` | 2025-09-29 | Extended thinking left off (parameter omitted) |

All three models use `temperature = 0`.

**LMprior is run on `gpt-4.1-2025-04-14` only.** It is the only model that returns
token-level log-probabilities: Anthropic exposes no such parameter, and the Gemini API
returns `Logprobs is not enabled for this model`. See `capability_test_result.json` for
the empirical verification.

## Note on reproducibility

The `seed` parameter is supported only by the OpenAI API. Anthropic and Google expose no
equivalent, so in this notebook `SEEDS` identifies **repeated runs**. API-level seed
fixing applies to GPT-4.1 only.

## Before running

```bash
pip install openai anthropic requests pandas matplotlib
```

API keys are configured in **Section 1** via a `.env` file or shell environment variables.
They are never written into the code.

## 0. Environment check

Run this first if a cell fails to execute. Most problems are visible in its output.

In [1]:
"""Environment check. Copy this output verbatim when reporting a problem."""
import sys, platform, os
from pathlib import Path

print("Python      :", sys.version.split()[0], f"({platform.system()} {platform.release()})")
print("Executable  :", sys.executable)
print("Working dir :", Path.cwd())
print()

print("Contents of working directory:")
for p in sorted(Path.cwd().iterdir())[:20]:
    print(f"   {'[D]' if p.is_dir() else '   '} {p.name}")
print()

print("Packages:")
for mod in ["pandas", "requests", "openai", "anthropic"]:
    try:
        m = __import__(mod)
        print(f"   {mod:<12s} {getattr(m, '__version__', 'version unknown')}")
    except ImportError:
        print(f"   {mod:<12s} NOT INSTALLED  ->  pip install {mod}")
print()

pool = Path("../1. feature pool generator/feature_pool.txt")
if pool.exists():
    n = len([l for l in pool.read_text(encoding="utf-8").splitlines() if l.strip()])
    print(f"feature_pool.txt : found - {n} lines  ({pool.resolve()})")
else:
    print(f"feature_pool.txt : NOT FOUND  (looked in: {pool.resolve()})")

print(".env             :", "present" if Path(".env").exists() else "absent")

Python      : 3.12.10 (Windows 11)
Executable  : C:\Users\gykwa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe
Working dir : c:\Users\gykwa\kyunghee2026\yonsei\곽가영_제출\2. feature selection experiment

Contents of working directory:
       .env
       .gitignore
       LLM_feature_selection_experiments.ipynb
   [D] ranked_groups
   [D] results

Packages:
   pandas       3.0.3
   requests     2.34.2
   openai       2.53.0
   anthropic    0.121.0

feature_pool.txt : found - 107 lines  (C:\Users\gykwa\kyunghee2026\yonsei\곽가영_제출\1. feature pool generator\feature_pool.txt)
.env             : present


In [2]:
import sys, os, jupyter_core.paths as p
print("exec      :", sys.executable)
print("prefix    :", sys.prefix)
print("base      :", sys.base_prefix)      # prefix와 같으면 가상환경 아님
print("CONDA     :", os.environ.get("CONDA_PREFIX"))
print("VIRTUALENV:", os.environ.get("VIRTUAL_ENV"))
print("kernels   :", p.jupyter_path("kernels"))

exec      : C:\Users\gykwa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe
prefix    : C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0
base      : C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0
CONDA     : None
VIRTUALENV: None
kernels   : ['C:\\Users\\gykwa\\AppData\\Roaming\\jupyter\\kernels', 'C:\\Users\\gykwa\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\share\\jupyter\\kernels', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\\share\\jupyter\\kernels']


## 1. API keys

Keys are **never written into the code**. Use one of the two methods below.

**Method A — `.env` file (recommended)**

Copy `.env.example` to `.env` in this folder and fill in the values.

```powershell
Copy-Item .env.example .env     # PowerShell
```
```bash
cp .env.example .env            # bash
```

**Method B — shell environment variables**

```powershell
$env:OPENAI_API_KEY="..."; $env:ANTHROPIC_API_KEY="..."; $env:GEMINI_API_KEY="..."
```

If both are present, **shell environment variables take precedence**.

> `.env` holds real credentials. It is listed in `.gitignore` so it will not be committed,
> but you must exclude it yourself when packaging files for submission.

In [3]:
"""Load API keys from a .env file or from shell environment variables.

Precedence: shell environment variables > .env file
(.env never overwrites a variable that is already set.)

If .env is missing, copy .env.example to .env, fill in the values, and re-run this cell.
"""
import os
from pathlib import Path

ENV_FILE = Path(".env")
REQUIRED = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]


def load_dotenv(path: Path = ENV_FILE) -> list[str]:
    """Read KEY=VALUE lines from .env into os.environ. Returns the keys actually set."""
    filled = []
    if not path.exists():
        return filled
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip()
        if len(v) >= 2 and v[0] == v[-1] and v[0] in "\"'":   # strip quotes
            v = v[1:-1]
        if not v:
            continue
        if k not in os.environ:      # shell wins; do not overwrite
            os.environ[k] = v
            filled.append(k)
    return filled


def _mask(v: str) -> str:
    return f"{v[:6]}...{v[-4:]}" if len(v) > 12 else "set"


_from_file = load_dotenv()
if ENV_FILE.exists():
    filled = ", ".join(_from_file) if _from_file else "none (all already set in shell)"
    print(f".env file : present  (keys taken from it: {filled})")
else:
    print(".env file : absent  (using shell environment variables only)")
print()

_missing = []
for name in REQUIRED:
    val = os.environ.get(name)
    src = ".env" if name in _from_file else "shell"
    if val:
        print(f"  {name:<20s} OK   [{src}]  {_mask(val)}")
    else:
        print(f"  {name:<20s} MISSING")
        _missing.append(name)

if _missing:
    raise RuntimeError(
        "The following keys are not set: " + ", ".join(_missing) + "\n"
        "  1) copy .env.example to .env\n"
        "  2) open .env and fill in the values\n"
        "  3) re-run this cell\n"
        "  (.env is listed in .gitignore, so it will not be committed. "
        "Exclude it manually when packaging files for submission.)"
    )
print("\nAll required keys present.")

.env file : present  (keys taken from it: OPENAI_API_KEY, ANTHROPIC_API_KEY, GEMINI_API_KEY)

  OPENAI_API_KEY       OK   [.env]  sk-pro...YQYA
  ANTHROPIC_API_KEY    OK   [.env]  sk-ant...xgAA
  GEMINI_API_KEY       OK   [.env]  AIzaSy...ocuE

All required keys present.


## 2. Experiment configuration

In [4]:
"""Experiment constants and model definitions.

Edit only this cell to change models, paths, or the number of repeated runs.
"""
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# ---- Input ------------------------------------------------------------------
# The relative path depends on where Jupyter was launched, so try candidates in order.
_POOL_CANDIDATES = [
    Path("../1. feature pool generator/feature_pool.txt"),
    Path("./1. feature pool generator/feature_pool.txt"),
    Path("곽가영_제출/1. feature pool generator/feature_pool.txt"),
]
FEATURE_POOL = next((p for p in _POOL_CANDIDATES if p.exists()), None)
if FEATURE_POOL is None:
    raise FileNotFoundError(
        "Could not locate feature_pool.txt.\n"
        f"  Working directory: {Path.cwd()}\n"
        "  Paths tried:\n" + "\n".join(f"    - {p.resolve()}" for p in _POOL_CANDIDATES) +
        "\n  -> Check that the notebook was opened inside the "
        "'2. feature selection experiment' folder, or add the correct path to "
        "_POOL_CANDIDATES above."
    )

# ---- Output -----------------------------------------------------------------
RESULT_DIR = Path("results")            # raw per-call records
AGG_DIR    = Path("results/aggregate")  # aggregated tables
for _d in (RESULT_DIR, AGG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---- Repeated runs ----------------------------------------------------------
# Passed to the OpenAI API as `seed`. For other vendors these are run identifiers only.
SEEDS = [42, 43, 44, 45, 46]

# ---- Call parameters --------------------------------------------------------
TEMPERATURE            = 0.0
TOP_P                  = 1.0
TIMEOUT                = 120     # seconds
MAX_RETRY              = 5       # attempts on validation failure or API error
RETRY_BACKOFF          = 2.0     # seconds, multiplied by the attempt number
GEMINI_THINKING_BUDGET = 128     # Gemini 2.5 Pro cannot disable thinking; use the minimum
N_RANK_GROUPS          = 3       # LLM-Rank group count. A single call omits features, so
                                 # the list is split and merged pairwise (see Section 4).
LMPRIOR_TOP_LOGPROBS   = 20      # API maximum. Lower values truncate log P(N) (Section 7).
LMPRIOR_FLOOR          = -100.0  # sentinel for an unobserved token (not a probability)


@dataclass(frozen=True)
class ModelSpec:
    key: str        # short identifier used in filenames and tables
    vendor: str     # openai | anthropic | google
    model_id: str   # exact snapshot ID sent to the API
    env_key: str    # environment variable holding the API key


MODELS = {
    "gpt":    ModelSpec("gpt",    "openai",    "gpt-4.1-2025-04-14",         "OPENAI_API_KEY"),
    "gemini": ModelSpec("gemini", "google",    "gemini-2.5-pro",             "GEMINI_API_KEY"),
    "claude": ModelSpec("claude", "anthropic", "claude-sonnet-4-5-20250929", "ANTHROPIC_API_KEY"),
}

LMPRIOR_MODEL = MODELS["gpt"]   # the only model that returns log-probabilities

# ---- Load the feature pool --------------------------------------------------
EXPECTED_N_FEATURES = 107   # produced by notebook 1 (feature pool generator)

FEATURES = [ln.strip() for ln in FEATURE_POOL.read_text(encoding="utf-8").splitlines()
            if ln.strip()]
N_FEATURES = len(FEATURES)

if N_FEATURES != EXPECTED_N_FEATURES:
    raise ValueError(
        f"Expected {EXPECTED_N_FEATURES} features but found {N_FEATURES}.\n"
        f"  File: {FEATURE_POOL.resolve()}\n"
        "  -> Re-run '1. Feature Pool Extraction.ipynb' to regenerate the file, or "
        "update EXPECTED_N_FEATURES above if the pool was changed deliberately."
    )
if len(set(FEATURES)) != N_FEATURES:
    dup = sorted({f for f in FEATURES if FEATURES.count(f) > 1})
    raise ValueError(f"Duplicate entries in the feature pool: {dup}")

print(f"Working dir  : {Path.cwd()}")
print(f"Feature pool : {N_FEATURES} features  ({FEATURE_POOL.resolve()})")
print(f"Models       : {', '.join(m.model_id for m in MODELS.values())}")
print(f"Repeats      : {SEEDS}")
print(f"Output dir   : {RESULT_DIR.resolve()}")

pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

Working dir  : c:\Users\gykwa\kyunghee2026\yonsei\곽가영_제출\2. feature selection experiment
Feature pool : 107 features  (C:\Users\gykwa\kyunghee2026\yonsei\곽가영_제출\1. feature pool generator\feature_pool.txt)
Models       : gpt-4.1-2025-04-14, gemini-2.5-pro, claude-sonnet-4-5-20250929
Repeats      : [42, 43, 44, 45, 46]
Output dir   : C:\Users\gykwa\kyunghee2026\yonsei\곽가영_제출\2. feature selection experiment\results


## 3. Shared utilities

Differences between the three APIs are absorbed by a single `chat()` function. Every
method in Sections 4–7 calls only this function, so prompts and computation logic cannot
diverge across vendors.

`chat()` returns the response text together with **token usage**. Recording Gemini's
`thoughtsTokenCount` allows the residual asymmetry in reasoning configuration to be
reported quantitatively.

In [5]:
"""Vendor adapter. Absorbs the differences between the three APIs."""

_clients: dict[str, object] = {}


def _api_key(spec: ModelSpec) -> str:
    key = os.environ.get(spec.env_key)
    if not key:
        raise RuntimeError(
            f"Environment variable {spec.env_key} is not set. "
            f"Set it in the shell or in .env, then restart the kernel."
        )
    return key


def _openai_client():
    if "openai" not in _clients:
        from openai import OpenAI
        _clients["openai"] = OpenAI(api_key=_api_key(MODELS["gpt"]))
    return _clients["openai"]


def _anthropic_client():
    if "anthropic" not in _clients:
        from anthropic import Anthropic
        _clients["anthropic"] = Anthropic(api_key=_api_key(MODELS["claude"]))
    return _clients["anthropic"]


def chat(spec: ModelSpec, system: str, user: str, *, max_tokens: int,
         seed: int | None = None) -> tuple[str, dict]:
    """Single-turn call. Returns (response text, usage dict).

    Usage dict keys: prompt_tokens, completion_tokens, thinking_tokens, model_returned
    """
    if spec.vendor == "openai":
        kwargs = dict(
            model=spec.model_id,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
            temperature=TEMPERATURE, top_p=TOP_P,
            max_tokens=max_tokens, timeout=TIMEOUT,
        )
        if seed is not None:
            kwargs["seed"] = seed          # supported by OpenAI only
        r = _openai_client().chat.completions.create(**kwargs)
        u = r.usage
        return r.choices[0].message.content or "", {
            "prompt_tokens": u.prompt_tokens,
            "completion_tokens": u.completion_tokens,
            "thinking_tokens": 0,
            "model_returned": r.model,
        }

    if spec.vendor == "anthropic":
        # The `thinking` parameter is deliberately omitted, leaving extended thinking off.
        r = _anthropic_client().messages.create(
            model=spec.model_id,
            max_tokens=max_tokens,
            temperature=TEMPERATURE,
            system=system,
            messages=[{"role": "user", "content": user}],
        )
        u = r.usage
        text = "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
        return text, {
            "prompt_tokens": u.input_tokens,
            "completion_tokens": u.output_tokens,
            "thinking_tokens": 0,
            "model_returned": r.model,
        }

    if spec.vendor == "google":
        # Gemini 2.5 Pro cannot disable thinking; thinkingBudget minimizes it.
        url = ("https://generativelanguage.googleapis.com/v1beta/models/"
               f"{spec.model_id}:generateContent")
        body = {
            "systemInstruction": {"parts": [{"text": system}]},
            "contents": [{"role": "user", "parts": [{"text": user}]}],
            "generationConfig": {
                "temperature": TEMPERATURE,
                "topP": TOP_P,
                "maxOutputTokens": max_tokens,
                "thinkingConfig": {"thinkingBudget": GEMINI_THINKING_BUDGET},
            },
        }
        resp = requests.post(url, params={"key": _api_key(spec)}, json=body, timeout=TIMEOUT)
        if resp.status_code != 200:
            raise RuntimeError(f"Gemini HTTP {resp.status_code}: {resp.text[:400]}")
        j = resp.json()
        cand = j["candidates"][0]
        text = "".join(p.get("text", "") for p in cand.get("content", {}).get("parts", []))
        um = j.get("usageMetadata", {})
        return text, {
            "prompt_tokens": um.get("promptTokenCount"),
            "completion_tokens": um.get("candidatesTokenCount"),
            "thinking_tokens": um.get("thoughtsTokenCount", 0),
            "model_returned": j.get("modelVersion", spec.model_id),
        }

    raise ValueError(f"Unknown vendor: {spec.vendor}")

In [6]:
"""JSON parsing, retry, and file I/O helpers."""


def parse_json(text: str):
    """Strip code fences and parse as JSON."""
    s = (text or "").strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]          # drop the opening fence line
        s = s.rsplit("```", 1)[0]         # drop the closing fence
    return json.loads(s.strip())

def call_validated(spec: ModelSpec, system: str, user: str, *, max_tokens: int,
                   validate=None, seed: int | None = None, label: str = ""):
    last = None
    base_user = user

    for attempt in range(1, MAX_RETRY + 1):
        raw, got_response = None, False
        try:
            raw, usage = chat(spec, system, user, max_tokens=max_tokens, seed=seed)
            got_response = True                    
            obj = parse_json(raw)
            if validate is not None:
                problem = validate(obj)
                if problem:
                    raise ValueError(problem)
            if attempt > 1:
                print(f"    -> {label} recovered on attempt {attempt}")
            return obj, usage

        except Exception as e:
            last = e
            print(f"    ! {label} attempt {attempt}/{MAX_RETRY} "
                  f"[{'content' if got_response else 'transport'}] "
                  f"failed: {str(e)[:150]}")
            if attempt >= MAX_RETRY:
                break

            if got_response:
                shown = (raw or "").strip().replace("\n", " ")
                if len(shown) > 400:
                    shown = shown[:400] + " ..."
                user = base_user + "\n" + "\n".join([
                    f"[CORRECTION {attempt}]",
                    f"Your previous response was: {shown or '(empty)'}",
                    f"It was rejected because: {e}",
                    "Fix exactly that problem and answer again. Reply with JSON only, "
                    "in the schema requested above. Do not write anything before or "
                    "after the JSON object.",
                ])
            else:
                user = base_user                   

            time.sleep(RETRY_BACKOFF * attempt)

    raise RuntimeError(f"{label}: failed after {MAX_RETRY} attempts. Last error: {last}")


def read_jsonl(path: Path) -> list[dict]:
    """Parse a JSONL file.

    Records are separated by "\\n" only. Python's str.splitlines() must NOT be used
    here: it also breaks on U+2028, U+2029 and U+0085, which are legal unescaped
    inside JSON strings and do occur in OpenAI logprob token values. Using
    splitlines() shatters such records into unparseable fragments.
    """
    rows, bad = [], []
    for i, ln in enumerate(path.read_text(encoding="utf-8").split("\n"), start=1):
        if not ln.strip():
            continue
        try:
            rows.append(json.loads(ln))
        except json.JSONDecodeError:
            bad.append(i)
    if bad:
        print(f"    ! {path}: {len(bad)} malformed line(s) at {bad[:5]}"
              f"{' ...' if len(bad) > 5 else ''} - {len(rows)} record(s) recovered")
    return rows

def write_jsonl(path: Path, rows: list[dict]) -> None:
    """Write atomically, escaping non-ASCII.

    ensure_ascii=True keeps U+2028/U+2029/U+0085 out of the raw bytes, so the file
    stays one-record-per-physical-line under any reader. JSON round-trips identically
    either way. The temp-file-then-replace pattern prevents a crash from leaving a
    half-written file in place of a complete one.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + "\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)
    print(f"    saved: {path}  ({len(rows)} rows)")


def out_path(method: str, model_key: str, seed: int) -> Path:
    return RESULT_DIR / method / model_key / f"seed{seed}.jsonl"


RUN_META = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "n_features": N_FEATURES,
    "feature_pool": str(FEATURE_POOL),
    "seeds": SEEDS,
    "temperature": TEMPERATURE,
    "n_rank_groups": N_RANK_GROUPS,
    "gemini_thinking_budget": GEMINI_THINKING_BUDGET,
    "models": {k: asdict(v) for k, v in MODELS.items()},
}
print("Run metadata initialized.")

Run metadata initialized.


## 4. LLM-Rank

The feature list is split into groups, each group is ranked, and the group rankings are
merged into a single ordering by **pairwise comparison** (`N_RANK_GROUPS = 3`).

### Why split the list

Presenting all features in one call causes the model to **omit** some of them. Splitting
the list reduces the number of items handled per call and largely eliminates omissions.
Each group response is still validated against its input set.

### Retry with correction

At `temperature = 0` an identical prompt yields an identical response, so repeating the
same request cannot fix a rejected output. When validation fails, the **specific reason**
(for example, 35 of 36 features returned) is appended to the prompt and the request is
reissued. The appended text concerns item count, naming, and output format only; it
carries no information about which features are important. After `MAX_RETRY` failures an
exception is raised.

### Merge procedure

The two ranked lists are traversed as in the merge step of merge sort. At each position
the two candidates are presented to the model, which selects the more important one. When
one list is exhausted the remainder of the other is appended unchanged.

```
i, j = 0, 0
while i < len(A) and j < len(B):
    winner = compare(A[i], B[j])
    append the winner and advance its pointer
result += A[i:] + B[j:]
```

The presentation order is reshuffled for each run, distributing the influence of group
composition and within-list position across repeats. The merged result is validated
against the full pool one final time.

### Files written

| File | Contents |
|---|---|
| `results/rank/{model}/seed{n}.jsonl` | Final merged ranking (used for aggregation) |
| `results/rank/{model}/seed{n}_groups.json` | Group composition and per-group rankings |
| `results/rank/{model}/seed{n}_comparisons.jsonl` | Pairwise comparison log (winner, reason) |

In [7]:
RANK_SYSTEM = """
You are a clinical feature selection assistant for ICU delirium prediction.

Your task:
- Rank the following clinical features from most important to least important for predicting ICU delirium.
- Only rank the features given below. Do not add, omit, or rename any features.
- Use the exact feature names provided without any modification (case, spacing, spelling, etc).

IMPORTANT:
- Do not omit any feature.
- Do not change the feature names in any way.
- Do not include explanations, text, or markdown.
- Respond strictly in JSON format only.

Response format:
["feature1", "feature2", "feature3", ...]
"""

COMPARE_SYSTEM = """
You are given two clinical features regarding ICU delirium prediction.

Your task:
- Determine which feature is more important for predicting ICU delirium.
- Respond strictly in JSON format with two keys.

IMPORTANT:
- Do not change the feature names in any way.
- Do not include explanations, text, or markdown.
- Only return raw JSON.

Response format:
{
  "winner": "A" or "B",
  "reason": "Short reason (1-2 sentences) why the selected feature is more important."
}
"""


def _validate_ranking(ranked, expected):
    """Check that the returned list is exactly the same set as the input."""
    if not isinstance(ranked, list):
        return f"not a list ({type(ranked).__name__})"
    if len(ranked) != len(expected):
        return f"count mismatch: {len(expected)} requested, {len(ranked)} returned"
    if len(set(ranked)) != len(ranked):
        dup = sorted({x for x in ranked if ranked.count(x) > 1})
        return f"duplicates: {dup[:5]}"
    missing = set(expected) - set(ranked)
    extra = set(ranked) - set(expected)
    if missing or extra:
        return f"name mismatch - missing {sorted(missing)[:5]} / extra {sorted(extra)[:5]}"
    return None


def _validate_compare(obj):
    if not isinstance(obj, dict):
        return f"not a dict ({type(obj).__name__})"
    w = str(obj.get("winner", "")).strip()
    if w not in ("A", "B"):
        return f"winner is not A or B: {obj.get('winner')!r}"
    return None


def split_groups(features, n_groups):
    """Split the list into n_groups near-equal chunks (earlier chunks may be larger)."""
    size = -(-len(features) // n_groups)          # ceiling division
    return [features[i:i + size] for i in range(0, len(features), size)][:n_groups]


def rank_group(spec, group, seed, label):
    """Rank one group, reissuing the request with a correction note on failure.

    At temperature=0 an identical prompt produces an identical response, so a plain retry
    cannot change the outcome. When the returned list does not match the input, the
    specific discrepancy is appended to the prompt. The appended text concerns item count,
    naming, and format only, and carries no information about feature importance.
    """
    base_user = ", ".join(group)
    user = base_user
    last_problem = None

    for attempt in range(1, MAX_RETRY + 1):
        try:
            text, usage = chat(spec, RANK_SYSTEM, user, max_tokens=8192, seed=seed)
            ranked = parse_json(text)
            problem = _validate_ranking(ranked, group)
            if problem is None:
                if attempt > 1:
                    print(f"    . {label} succeeded on attempt {attempt}")
                return ranked, usage
            last_problem = problem
        except Exception as e:
            last_problem = f"{type(e).__name__}: {e}"

        print(f"    ! {label} attempt {attempt}/{MAX_RETRY} failed: {str(last_problem)[:170]}")

        if attempt < MAX_RETRY:
            # Change the prompt so the response can differ.
            user = (
                f"{base_user}\n\n"
                f"[CORRECTION] Your previous response was rejected: {last_problem}\n"
                f"Return ALL {len(group)} features listed above - every one exactly once, "
                f"with names copied verbatim - as a single flat JSON array. "
                f"Count them before responding."
            )
            time.sleep(RETRY_BACKOFF * attempt)

    raise RuntimeError(f"{label}: failed after {MAX_RETRY} attempts. Last reason: {last_problem}")


def merge_two(spec, A, B, seed, label):
    """Merge two ranked lists by pairwise comparison. Returns (merged, comparison log)."""
    merged, log = [], []
    i = j = 0
    while i < len(A) and j < len(B):
        obj, usage = call_validated(
            spec, COMPARE_SYSTEM, f"A: {A[i]}\nB: {B[j]}\n",
            max_tokens=300, validate=_validate_compare,
            seed=seed, label=f"{label}/cmp{len(log) + 1}",
        )
        winner = str(obj["winner"]).strip()
        log.append({"step": len(log) + 1, "feature_A": A[i], "feature_B": B[j],
                    "winner": winner, "reason": obj.get("reason", ""),
                    "prompt_tokens": usage["prompt_tokens"],
                    "completion_tokens": usage["completion_tokens"],
                    "thinking_tokens": usage["thinking_tokens"]})
        if winner == "A":
            merged.append(A[i]); i += 1
        else:
            merged.append(B[j]); j += 1
    merged.extend(A[i:])          # the remainder is already ordered
    merged.extend(B[j:])
    return merged, log


def run_rank(spec, seed):
    """Returns (final ranking rows, group info dict, comparison log)."""
    rng = random.Random(seed)
    shuffled = FEATURES.copy()
    rng.shuffle(shuffled)

    groups = split_groups(shuffled, N_RANK_GROUPS)
    print(f"    group sizes: {[len(g) for g in groups]}")

    ranked_groups, usages = [], []
    for gi, g in enumerate(groups):
        r, u = rank_group(spec, g, seed, f"rank/{spec.key}/seed{seed}/group{gi}")
        ranked_groups.append(r)
        usages.append(u)
        print(f"    group {gi} ranked ({len(r)} features)")

    merged, all_log = ranked_groups[0], []
    for gi in range(1, len(ranked_groups)):
        merged, log = merge_two(spec, merged, ranked_groups[gi], seed,
                                f"rank/{spec.key}/seed{seed}/merge{gi}")
        all_log += log
        print(f"    merge {gi} done ({len(log)} comparisons)")

    problem = _validate_ranking(merged, FEATURES)
    if problem:
        raise RuntimeError(f"rank/{spec.key}/seed{seed} merged result invalid: {problem}")

    rows = [{"rank": i, "feature": f, "seed": seed,
             "model": spec.model_id, "model_returned": usages[0]["model_returned"],
             "prompt_tokens": usages[0]["prompt_tokens"],
             "completion_tokens": usages[0]["completion_tokens"],
             "thinking_tokens": usages[0]["thinking_tokens"]}
            for i, f in enumerate(merged, start=1)]

    groups_obj = {"seed": seed, "model": spec.model_id, "n_groups": N_RANK_GROUPS,
                  "shuffled": shuffled, "groups": groups, "ranked_groups": ranked_groups}
    return rows, groups_obj, all_log


# ---- Run --------------------------------------------------------------------
for mkey, spec in MODELS.items():
    print(f"\n[LLM-Rank] {spec.model_id}  ({N_RANK_GROUPS} groups)")
    for seed in SEEDS:
        p = out_path("rank", mkey, seed)
        if p.exists():
            print(f"  seed {seed}: existing result reused ({p})")
            continue
        print(f"  seed {seed} ...")
        rows, groups_obj, log = run_rank(spec, seed)
        write_jsonl(p, rows)
        (p.parent / f"seed{seed}_groups.json").write_text(
            json.dumps(groups_obj, ensure_ascii=False, indent=2), encoding="utf-8")
        write_jsonl(p.parent / f"seed{seed}_comparisons.jsonl", log)


[LLM-Rank] gpt-4.1-2025-04-14  (3 groups)
  seed 42: existing result reused (results\rank\gpt\seed42.jsonl)
  seed 43: existing result reused (results\rank\gpt\seed43.jsonl)
  seed 44: existing result reused (results\rank\gpt\seed44.jsonl)
  seed 45: existing result reused (results\rank\gpt\seed45.jsonl)
  seed 46: existing result reused (results\rank\gpt\seed46.jsonl)

[LLM-Rank] gemini-2.5-pro  (3 groups)
  seed 42: existing result reused (results\rank\gemini\seed42.jsonl)
  seed 43: existing result reused (results\rank\gemini\seed43.jsonl)
  seed 44: existing result reused (results\rank\gemini\seed44.jsonl)
  seed 45: existing result reused (results\rank\gemini\seed45.jsonl)
  seed 46: existing result reused (results\rank\gemini\seed46.jsonl)

[LLM-Rank] claude-sonnet-4-5-20250929  (3 groups)
  seed 42: existing result reused (results\rank\claude\seed42.jsonl)
  seed 43: existing result reused (results\rank\claude\seed43.jsonl)
  seed 44: existing result reused (results\rank\claude

In [8]:
"""LLM-Rank aggregation: mean and standard deviation of rank across repeats."""

def aggregate_rank(model_key: str) -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        p = out_path("rank", model_key, seed)
        if not p.exists():
            print(f"  missing: {p}")
            continue
        rows += read_jsonl(p)
    if not rows:
        return pd.DataFrame()
    d = pd.DataFrame(rows)
    return (d.groupby("feature")["rank"]
              .agg(avg_rank="mean", std_rank="std", n_seed="count")
              .reset_index()
              .sort_values("avg_rank")
              .reset_index(drop=True))


rank_results = {}
for mkey in MODELS:
    df = aggregate_rank(mkey)
    if df.empty:
        continue
    rank_results[mkey] = df
    df.to_csv(AGG_DIR / f"rank_{mkey}.csv", index=False, encoding="utf-8-sig")
    print(f"rank/{mkey}: {len(df)} features")

rank_results.get("gpt", pd.DataFrame()).head(15)

rank/gpt: 107 features
rank/gemini: 107 features
rank/claude: 107 features


,feature,avg_rank,std_rank,n_seed
0,Dementia,1.8,1.303840,5
1,Glasgow Coma Scale,2.4,1.140175,5
2,Sequential Organ Failure Assessment Score,3.8,3.563706,5
3,Acute Physiology and Chronic Health Evaluation (APACHE) Score,4.8,2.683282,5
4,Simplified Acute Physiology Score II,7.6,4.505552,5
5,Mechanical Ventilation,7.8,3.114482,5
6,Vasopressor Use,7.8,1.923538,5
7,Sedation Use,8.0,3.316625,5
8,Sepsis,9.2,3.114482,5
9,Use Of Midazolam,9.4,3.577709,5


## 5. LLM-Score

Each feature is presented individually and an importance score in [0, 1] is requested.
Scores are averaged per feature across repeats.

Requiring the score as a **string with five decimal places** discourages the model from
clustering responses on rounded values such as 0.8 or 0.9 (round-number bias).

In [9]:
SCORE_SYSTEM = """
You are a clinical feature selection assistant for ICU delirium prediction.

You are given:
- A list of clinical features for evaluation.
Your task:
- For **each feature**, provide an importance score (between 0 and 1; with higher values indicating greater importance) for predicting ICU delirium.
- Include a concise clinical justification for the score.

IMPORTANT:
- Base your reasoning on clinical relevance and associations with ICU delirium.
- Each score must be a string formatted to five decimal places (e.g., "0.87341").
- Respond strictly in JSON format only.
- Do not include any explanations, text, or markdown.
- Do not use backticks or extra formatting.

Response format:
{
  "feature": "Feature Name",
  "score": "Score",
  "reason": "Short explanation"
}
"""


def _unwrap_single(obj):
    """Some models wrap a single JSON object in a one-element array.

    The prompt says "a list of clinical features ... for each feature", but the user
    message carries exactly one feature. GPT-4.1 and Claude return a bare object;
    Gemini follows the wording literally and returns [{...}]. The prompt is left
    unchanged so that runs already completed remain comparable, and the difference is
    absorbed here at the parsing layer. Content is identical either way.
    """
    if isinstance(obj, list) and len(obj) == 1 and isinstance(obj[0], dict):
        return obj[0]
    return obj


def _validate_score(obj):
    obj = _unwrap_single(obj)
    if not isinstance(obj, dict):
        return f"not a dict ({type(obj).__name__})"
    if "score" not in obj:
        return "missing 'score' key"
    try:
        v = float(obj["score"])
    except (TypeError, ValueError):
        return f"score is not convertible to float: {obj.get('score')!r}"
    if not (0.0 <= v <= 1.0):
        return f"score outside [0, 1]: {v}"
    return None


def run_score(spec, seed: int) -> list[dict]:
    rows = []
    for i, feature in enumerate(FEATURES, start=1):
        obj, usage = call_validated(
            spec, SCORE_SYSTEM, f'Feature: "{feature}"\n',
            max_tokens=4096, validate=_validate_score, seed=seed,
            label=f"score/{spec.key}/seed{seed}/{feature}",
        )
        obj = _unwrap_single(obj)               # normalize before reading fields
        rows.append({
            "feature": feature,                 # trust the input name, not the echo
            "feature_returned": obj.get("feature"),
            "score": float(obj["score"]),
            "reason": obj.get("reason", ""),
            "seed": seed, "model": spec.model_id,
            "model_returned": usage["model_returned"],
            "prompt_tokens": usage["prompt_tokens"],
            "completion_tokens": usage["completion_tokens"],
            "thinking_tokens": usage["thinking_tokens"],
        })
        if i % 25 == 0 or i == len(FEATURES):
            print(f"    {i}/{len(FEATURES)}")
    return rows


for mkey, spec in MODELS.items():
    print(f"\n[LLM-Score] {spec.model_id}")
    for seed in SEEDS:
        p = out_path("score", mkey, seed)
        if p.exists():
            print(f"  seed {seed}: existing result reused")
            continue
        print(f"  seed {seed} ...")
        write_jsonl(p, run_score(spec, seed))


[LLM-Score] gpt-4.1-2025-04-14
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused

[LLM-Score] gemini-2.5-pro
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused

[LLM-Score] claude-sonnet-4-5-20250929
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused


In [10]:
"""LLM-Score aggregation: mean score per feature."""

def aggregate_score(model_key: str) -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        p = out_path("score", model_key, seed)
        if p.exists():
            rows += read_jsonl(p)
    if not rows:
        return pd.DataFrame()
    d = pd.DataFrame(rows)
    return (d.groupby("feature")["score"]
              .agg(avg_score="mean", std_score="std", n_seed="count")
              .reset_index()
              .sort_values("avg_score", ascending=False)
              .reset_index(drop=True))


score_results = {}
for mkey in MODELS:
    df = aggregate_score(mkey)
    if df.empty:
        continue
    score_results[mkey] = df
    df.to_csv(AGG_DIR / f"score_{mkey}.csv", index=False, encoding="utf-8-sig")
    print(f"score/{mkey}: {len(df)} features")

score_results.get("gpt", pd.DataFrame()).head(15)

score/gpt: 107 features
score/gemini: 107 features
score/claude: 107 features


,feature,avg_score,std_score,n_seed
0,Dementia,0.9500,0.000000,5
1,Sedation Use,0.9140,0.008944,5
2,Glasgow Coma Scale,0.9108,0.001095,5
3,Use Of Midazolam,0.9100,0.000000,5
4,Mechanical Ventilation,0.8940,0.021909,5
5,Sepsis,0.8900,0.028284,5
6,Sequential Organ Failure Assessment Score,0.8720,0.000000,5
7,Infection,0.8500,0.000000,5
8,Age,0.8500,0.000000,5
9,Acute Physiology and Chronic Health Evaluation (APACHE) Score,0.8360,0.013416,5


## 6. LLM-Seq

Greedy forward selection. The model is shown the features already selected together with
the remaining candidates and selects the single feature to add next. The loop runs until
the candidate set is exhausted, so each repeat makes `N_FEATURES` calls.

The selection order is the ranking. For aggregation, the feature selected at step `t`
receives `N_FEATURES - t + 1` points, averaged across repeats.

In [11]:
SEQ_SYSTEM = """
You are a clinical feature selection assistant for ICU delirium prediction.

You are given:
- A list of features already selected.
- A list of remaining candidate features.

Your task:
- Select the **single most promising feature** from the candidate list that, when added to the selected features, is most likely to improve the model's AUROC for predicting ICU delirium.
- Base your decision on clinical knowledge and feature interactions.

IMPORTANT:
- Do **not** suggest features already selected.
- Do **not** change the feature names in any way.
- Respond strictly in JSON format only.
- Do **not** include explanations, text, or markdown.
- Do **not** suggest, abbreviate, or substitute any feature that is not listed in the provided candidate list.

Response format:
{
  "feature": "Feature Name",
  "reason": "Short explanation"
}
"""


def run_seq(spec, seed: int) -> list[dict]:
    selected: list[str] = []
    rows: list[dict] = []

    for step in range(1, len(FEATURES) + 1):
        remaining = [f for f in FEATURES if f not in selected]
        if not remaining:
            break

        def _validate(obj, _rem=remaining):
            if not isinstance(obj, dict):
                return f"not a dict ({type(obj).__name__})"
            f = obj.get("feature")
            if f not in _rem:
                return f"returned a feature not in the candidate list: {f!r}"
            return None

        user = (f'Features in use: "{selected}"\n'
                f'Candidates: "{remaining}"\n')
        obj, usage = call_validated(
            spec, SEQ_SYSTEM, user, max_tokens=4096, validate=_validate,
            seed=seed, label=f"seq/{spec.key}/seed{seed}/step{step}",
        )

        feature = obj["feature"]
        selected.append(feature)
        rows.append({
            "step": step, "feature": feature, "reason": obj.get("reason", ""),
            "seed": seed, "model": spec.model_id,
            "model_returned": usage["model_returned"],
            "prompt_tokens": usage["prompt_tokens"],
            "completion_tokens": usage["completion_tokens"],
            "thinking_tokens": usage["thinking_tokens"],
        })
        if step % 25 == 0 or step == len(FEATURES):
            print(f"    {step}/{len(FEATURES)}")

    assert len(rows) == len(FEATURES), f"selection count mismatch: {len(rows)} != {len(FEATURES)}"
    return rows


for mkey, spec in MODELS.items():
    print(f"\n[LLM-Seq] {spec.model_id}")
    for seed in SEEDS:
        p = out_path("seq", mkey, seed)
        if p.exists():
            print(f"  seed {seed}: existing result reused")
            continue
        print(f"  seed {seed} ...")
        write_jsonl(p, run_seq(spec, seed))


[LLM-Seq] gpt-4.1-2025-04-14
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused

[LLM-Seq] gemini-2.5-pro
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused

[LLM-Seq] claude-sonnet-4-5-20250929
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused


In [12]:
# [수정] 'import datetime' 는 §2 의 `from datetime import datetime, timezone` 로
#   바인딩된 datetime 클래스를 모듈로 덮어쓴다. 그 결과 뒤쪽 셀(§8 run_metadata)의
#   datetime.now(timezone.utc) 가 AttributeError 로 죽는다.
#   클래스 바인딩을 유지하고 functools 만 가져온다.
import functools

GEMINI_DEBUG_DIR = RESULT_DIR / "_debug"
GEMINI_FAILURES = []          # 실패 요약이 여기 쌓인다

class GeminiEmptyText(RuntimeError):
    """Gemini가 텍스트 파트 없이 응답한 경우."""

_orig_chat = chat             # 원본 보관

def chat(spec, system, user, *, max_tokens, seed=None):
    """google 경로만 진단 버전으로 교체. 나머지 벤더는 원본 그대로."""
    if spec.vendor != "google":
        return _orig_chat(spec, system, user, max_tokens=max_tokens, seed=seed)

    url = ("https://generativelanguage.googleapis.com/v1beta/models/"
           f"{spec.model_id}:generateContent")
    body = {
        "systemInstruction": {"parts": [{"text": system}]},
        "contents": [{"role": "user", "parts": [{"text": user}]}],
        "generationConfig": {
            "temperature": TEMPERATURE,
            "topP": TOP_P,
            "maxOutputTokens": max_tokens,
            "thinkingConfig": {"thinkingBudget": GEMINI_THINKING_BUDGET},
        },
    }
    resp = requests.post(url, params={"key": _api_key(spec)}, json=body, timeout=TIMEOUT)
    if resp.status_code != 200:
        raise RuntimeError(f"Gemini HTTP {resp.status_code}: {resp.text[:400]}")

    j = resp.json()
    um = j.get("usageMetadata", {})
    usage = {
        "prompt_tokens": um.get("promptTokenCount"),
        "completion_tokens": um.get("candidatesTokenCount"),
        "thinking_tokens": um.get("thoughtsTokenCount", 0),
        "model_returned": j.get("modelVersion", spec.model_id),
    }

    cand = (j.get("candidates") or [{}])[0]
    parts = (cand.get("content") or {}).get("parts") or []
    # thought=True 인 '생각' 파트는 본문이 아니므로 제외한다
    text = "".join(p.get("text", "") for p in parts
                   if isinstance(p, dict) and not p.get("thought"))

    if not text.strip():
        GEMINI_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%H%M%S_%f")
        (GEMINI_DEBUG_DIR / f"empty_{stamp}.json").write_text(
            json.dumps({"request": body, "response": j}, ensure_ascii=False, indent=2),
            encoding="utf-8")
        info = {
            "finishReason": cand.get("finishReason"),
            "n_parts": len(parts),
            "promptTok": usage["prompt_tokens"],
            "thoughtsTok": usage["thinking_tokens"],
            "candTok": usage["completion_tokens"],
            "maxOut": max_tokens,
            "budget": GEMINI_THINKING_BUDGET,
            "userChars": len(user),
        }
        GEMINI_FAILURES.append(info)
        raise GeminiEmptyText(" ".join(f"{k}={v}" for k, v in info.items()))

    return text, usage

print("Gemini 진단 모드 활성화 ->", GEMINI_DEBUG_DIR)

Gemini 진단 모드 활성화 -> results\_debug


In [13]:
"""LLM-Seq aggregation: selection order converted to points and averaged."""

def aggregate_seq(model_key: str) -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        p = out_path("seq", model_key, seed)
        if p.exists():
            rows += read_jsonl(p)
    if not rows:
        return pd.DataFrame()
    d = pd.DataFrame(rows)
    d["points"] = N_FEATURES - d["step"] + 1      # step t -> N - t + 1 points
    return (d.groupby("feature")
              .agg(avg_step=("step", "mean"), std_step=("step", "std"),
                   avg_points=("points", "mean"), n_seed=("step", "count"))
              .reset_index()
              .sort_values("avg_step")
              .reset_index(drop=True))


seq_results = {}
for mkey in MODELS:
    df = aggregate_seq(mkey)
    if df.empty:
        continue
    seq_results[mkey] = df
    df.to_csv(AGG_DIR / f"seq_{mkey}.csv", index=False, encoding="utf-8-sig")
    print(f"seq/{mkey}: {len(df)} features")

seq_results.get("gpt", pd.DataFrame()).head(15)

seq/gpt: 107 features
seq/gemini: 107 features
seq/claude: 107 features


,feature,avg_step,std_step,avg_points,n_seed
0,Glasgow Coma Scale,1.0,0.000000,107.0,5
1,Age,2.0,0.000000,106.0,5
2,Sedation Use,3.0,0.000000,105.0,5
3,Mechanical Ventilation,4.0,0.000000,104.0,5
4,Sequential Organ Failure Assessment Score,5.0,0.000000,103.0,5
5,Dementia,6.0,0.000000,102.0,5
6,Sepsis,7.0,0.000000,101.0,5
7,Alcohol Abuse,8.0,0.000000,100.0,5
8,Liver Disease,9.0,0.000000,99.0,5
9,Vasopressor Use,10.0,0.000000,98.0,5


## 7. LMprior — GPT-4.1 only

Each feature is presented with a Y/N question about whether it is important for delirium
prediction, and the score is computed from the log-probabilities of the response tokens:

$$\text{score} = \log P(\text{Y}) - \log P(\text{N})$$

### Why `top_logprobs = 20`

When `top_logprobs` is small and P(Y) is close to 1, the `N` token falls outside the
returned list and is **not observed**. `log P(N)` is then fixed at the sentinel value and
the score piles up against an artificial ceiling. This is referred to as **floor
truncation**, and the API maximum of 20 is used to mitigate it.

Unobserved tokens are recorded with the `y_observed` / `n_observed` flags. The substituted
`LMPRIOR_FLOOR` is a **missingness marker, not a probability**. Whether truncation remains
is checked in the cell that follows.

### Why this method uses GPT-4.1 alone

API testing (`capability_test_result.json`) found that token-level log-probabilities are
returned only by non-reasoning OpenAI models. Anthropic exposes no such parameter, and the
Gemini API returns `Logprobs is not enabled for this model`.

In [14]:
LMPRIOR_SYSTEM = """
You are a clinical feature selection assistant for ICU delirium prediction.

You are given:
- A single candidate clinical feature and its description.

Your task:
- Determine whether this feature is important for predicting ICU delirium in ICU patients.

IMPORTANT:
- Base your answer on clinical knowledge and feature relevance.
- Respond strictly in JSON format only.
- Do not include explanations, text, or markdown.

Response format:
{
  "feature": "Y/N",
  "reason": "Short explanation"
}
"""


def run_lmprior(seed: int) -> list[dict]:
    spec = LMPRIOR_MODEL
    client = _openai_client()
    rows = []

    for i, feature in enumerate(FEATURES, start=1):
        last_err = None
        for attempt in range(1, MAX_RETRY + 1):
            try:
                r = client.chat.completions.create(
                    model=spec.model_id,
                    messages=[{"role": "system", "content": LMPRIOR_SYSTEM},
                              {"role": "user", "content": f'Feature: "{feature}"\n'}],
                    temperature=TEMPERATURE, top_p=TOP_P, max_tokens=4096,
                    presence_penalty=0.0, frequency_penalty=0.0,
                    seed=seed, logprobs=True, top_logprobs=LMPRIOR_TOP_LOGPROBS,
                    timeout=TIMEOUT,
                )
                obj = parse_json(r.choices[0].message.content)

                # Scan the top_logprobs of every response token for the Y and N tokens.
                log_Y = log_N = None
                serial = []
                for tok in r.choices[0].logprobs.content:
                    for item in tok.top_logprobs:
                        serial.append({"token": item.token, "logprob": item.logprob})
                        clean = item.token.strip().strip('"')
                        if clean == "Y" and log_Y is None:
                            log_Y = item.logprob
                        elif clean == "N" and log_N is None:
                            log_N = item.logprob

                y_obs, n_obs = log_Y is not None, log_N is not None
                log_Y = LMPRIOR_FLOOR if log_Y is None else log_Y
                log_N = LMPRIOR_FLOOR if log_N is None else log_N

                u = r.usage
                rows.append({
                    "feature": feature,
                    "answer": obj.get("feature"),
                    "reason": obj.get("reason", ""),
                    "log_p_Y": log_Y, "log_p_N": log_N,
                    "y_observed": y_obs, "n_observed": n_obs,
                    "score": log_Y - log_N,
                    "logprobs": serial,
                    "seed": seed, "model": spec.model_id, "model_returned": r.model,
                    "prompt_tokens": u.prompt_tokens,
                    "completion_tokens": u.completion_tokens,
                    "thinking_tokens": 0,
                })
                break
            except Exception as e:
                last_err = e
                print(f"    ! lmprior/seed{seed}/{feature} attempt {attempt}/{MAX_RETRY}: {str(e)[:140]}")
                if attempt < MAX_RETRY:
                    time.sleep(RETRY_BACKOFF * attempt)
        else:
            raise RuntimeError(f"lmprior/seed{seed}/{feature}: retries exhausted - {last_err}")

        if i % 25 == 0 or i == len(FEATURES):
            print(f"    {i}/{len(FEATURES)}")

    return rows


print(f"\n[LMprior] {LMPRIOR_MODEL.model_id} (this model only)")
for seed in SEEDS:
    p = out_path("lmprior", LMPRIOR_MODEL.key, seed)
    if p.exists():
        print(f"  seed {seed}: existing result reused")
        continue
    print(f"  seed {seed} ...")
    write_jsonl(p, run_lmprior(seed))


[LMprior] gpt-4.1-2025-04-14 (this model only)
  seed 42: existing result reused
  seed 43: existing result reused
  seed 44: existing result reused
  seed 45: existing result reused
  seed 46: existing result reused


In [15]:
"""Floor-truncation check. Zero observation failures means top_logprobs=20 resolved it."""

_rows = []
for seed in SEEDS:
    p = out_path("lmprior", LMPRIOR_MODEL.key, seed)
    if p.exists():
        _rows += read_jsonl(p)

if _rows:
    d = pd.DataFrame(_rows)
    n_bad = int((~d["n_observed"]).sum())
    n_ceil = int((d["score"] == -LMPRIOR_FLOOR).sum())
    avg = d.groupby("feature")["score"].mean()
    print(f"Total responses          : {len(d)}")
    print(f"  log P(N) not observed  : {n_bad} ({n_bad / len(d) * 100:.1f}%)")
    print(f"  score at ceiling ({-LMPRIOR_FLOOR:.0f})  : {n_ceil}")
    print(f"  features tied at max   : {int((avg == avg.max()).sum())}")
    print(f"  score range            : {d['score'].min():.2f} to {d['score'].max():.2f}")
else:
    print("No LMprior results found.")

Total responses          : 535
  log P(N) not observed  : 0 (0.0%)
  score at ceiling (100)  : 0
  features tied at max   : 1
  score range            : -38.79 to 10.13


In [16]:
"""LMprior aggregation: mean rank across repeats."""

def aggregate_lmprior() -> pd.DataFrame:
    per_seed = []
    for seed in SEEDS:
        p = out_path("lmprior", LMPRIOR_MODEL.key, seed)
        if not p.exists():
            continue
        d = pd.DataFrame(read_jsonl(p))[["feature", "score"]]
        d[f"rank_{seed}"] = d["score"].rank(ascending=False, method="min")
        per_seed.append(d[["feature", f"rank_{seed}"]])
    if not per_seed:
        return pd.DataFrame()

    merged = per_seed[0]
    for d in per_seed[1:]:
        merged = merged.merge(d, on="feature")
    cols = [c for c in merged.columns if c.startswith("rank_")]
    merged["avg_rank"] = merged[cols].mean(axis=1)
    merged["std_rank"] = merged[cols].std(axis=1)
    return (merged[["feature", "avg_rank", "std_rank"]]
            .sort_values("avg_rank").reset_index(drop=True))


lmprior_result = aggregate_lmprior()
if not lmprior_result.empty:
    lmprior_result.to_csv(AGG_DIR / "lmprior_gpt.csv", index=False, encoding="utf-8-sig")
    print(f"lmprior/gpt: {len(lmprior_result)} features")
lmprior_result.head(15)

lmprior/gpt: 107 features


,feature,avg_rank,std_rank
0,Use Of Midazolam,1.6,0.894427
1,Sedation Use,1.6,0.547723
2,Mechanical Ventilation,3.4,1.140175
3,Sequential Organ Failure Assessment Score,5.4,1.341641
4,Alcohol Abuse,6.4,2.073644
5,Dementia,7.0,4.527693
6,Vasopressor Use,7.0,1.581139
7,Systemic Inflammatory Response Syndrome,7.0,4.062019
8,Age,9.6,2.190890
9,Sepsis,10.2,1.483240


## 8. Combined results

Results from the four methods and three models are combined into a single table. Every
cell is converted to a **rank** (1 = most important) so that the columns are comparable.

| Method | Native output | Conversion |
|---|---|---|
| LLM-Rank | Mean rank | Used directly (ascending) |
| LLM-Score | Mean score | Score descending |
| LLM-Seq | Mean selection step | Used directly (ascending) |
| LMprior | Mean rank | Used directly (ascending) |

In [17]:
"""Combine per-method, per-model rankings into one table."""

columns = {}

for mkey, df in rank_results.items():
    columns[f"rank_{mkey}"] = df.set_index("feature")["avg_rank"].rank(method="min")

for mkey, df in score_results.items():
    columns[f"score_{mkey}"] = df.set_index("feature")["avg_score"].rank(
        ascending=False, method="min")

for mkey, df in seq_results.items():
    columns[f"seq_{mkey}"] = df.set_index("feature")["avg_step"].rank(method="min")

if not lmprior_result.empty:
    columns["lmprior_gpt"] = lmprior_result.set_index("feature")["avg_rank"].rank(method="min")

combined = pd.DataFrame(columns)
combined = combined.reindex(FEATURES)                  # pool order; missing entries are NaN
combined["mean_rank"] = combined.mean(axis=1, skipna=True)
combined = combined.sort_values("mean_rank").reset_index().rename(columns={"index": "feature"})

combined.to_csv(AGG_DIR / "combined_ranks.csv", index=False, encoding="utf-8-sig")
print(f"Combined table saved: {AGG_DIR / 'combined_ranks.csv'} "
      f"({len(combined)} rows x {combined.shape[1]} columns)")
combined.head(20)

Combined table saved: results\aggregate\combined_ranks.csv (107 rows x 12 columns)


,feature,rank_gpt,rank_gemini,rank_claude,score_gpt,score_gemini,score_claude,seq_gpt,seq_gemini,seq_claude,lmprior_gpt,mean_rank
0,Sedation Use,8.0,4.0,6.0,2.0,2.0,1.0,3.0,4.0,3.0,1.0,3.4
1,Glasgow Coma Scale,2.0,1.0,3.0,3.0,2.0,3.0,1.0,1.0,2.0,18.0,3.6
2,Sequential Organ Failure Assessment Score,3.0,5.0,5.0,7.0,9.0,4.0,5.0,3.0,6.0,4.0,5.1
3,Dementia,1.0,2.0,7.0,1.0,2.0,2.0,6.0,12.0,12.0,6.0,5.1
4,Mechanical Ventilation,6.0,11.0,8.0,5.0,2.0,5.0,4.0,5.0,4.0,3.0,5.3
5,Age,11.0,8.0,14.0,8.0,2.0,19.0,2.0,2.0,1.0,9.0,7.6
6,Sepsis,9.0,12.0,10.0,6.0,1.0,5.0,7.0,9.0,8.0,10.0,7.7
7,Acute Physiology and Chronic Health Evaluation (APACHE) Score,4.0,3.0,4.0,10.0,25.0,8.0,14.0,6.0,5.0,12.0,9.1
8,Use Of Midazolam,10.0,6.0,13.0,4.0,2.0,7.0,17.0,11.0,20.0,1.0,9.1
9,Vasopressor Use,6.0,8.0,11.0,12.0,17.0,16.0,10.0,8.0,9.0,6.0,10.3


In [18]:
"""Quantify the residual asymmetry in reasoning configuration across models.

Gemini 2.5 Pro cannot disable thinking, so thinkingBudget=128 was used to minimize it.
Recording how many thinking tokens were actually consumed allows this to be reported
numerically rather than asserted.
"""
usage_rows = []
for path in (p for p in RESULT_DIR.rglob("seed*.jsonl") if "_comparisons" not in p.name):
    method, mkey = path.parts[-3], path.parts[-2]
    for r in read_jsonl(path):
        usage_rows.append({
            "method": method, "model_key": mkey, "model": r.get("model"),
            "prompt_tokens": r.get("prompt_tokens") or 0,
            "completion_tokens": r.get("completion_tokens") or 0,
            "thinking_tokens": r.get("thinking_tokens") or 0,
        })

if usage_rows:
    u = pd.DataFrame(usage_rows)
    summary = (u.groupby(["model", "method"])
                 .agg(n_calls=("prompt_tokens", "size"),
                      prompt_tokens=("prompt_tokens", "sum"),
                      completion_tokens=("completion_tokens", "sum"),
                      thinking_tokens=("thinking_tokens", "sum"),
                      mean_thinking_per_call=("thinking_tokens", "mean"))
                 .reset_index())
    summary.to_csv(AGG_DIR / "token_usage.csv", index=False, encoding="utf-8-sig")
    print("Token usage summary saved:", AGG_DIR / "token_usage.csv")
    display(summary)
else:
    print("No usage records found.")

Token usage summary saved: results\aggregate\token_usage.csv


,model,method,n_calls,prompt_tokens,completion_tokens,thinking_tokens,mean_thinking_per_call
0,claude-sonnet-4-5-20250929,rank,535,214963,180295,0,0.000000
1,claude-sonnet-4-5-20250929,score,535,115780,55951,0,0.000000
2,claude-sonnet-4-5-20250929,seq,535,607871,39306,0,0.000000
3,gemini-2.5-pro,rank,535,158467,150656,36273,67.800000
4,gemini-2.5-pro,score,535,104610,41970,47222,88.265421
5,gemini-2.5-pro,seq,535,411955,35008,82526,154.254206
6,gpt-4.1-2025-04-14,lmprior,535,63720,22958,0,0.000000
7,gpt-4.1-2025-04-14,rank,535,173768,122729,0,0.000000
8,gpt-4.1-2025-04-14,score,535,99565,29218,0,0.000000
9,gpt-4.1-2025-04-14,seq,535,440310,28604,0,0.000000


In [19]:
"""Save run metadata for the supplementary materials."""

RUN_META["finished_at"] = datetime.now(timezone.utc).isoformat()
RUN_META["result_files"] = sorted(str(p.relative_to(RESULT_DIR)) for p in RESULT_DIR.rglob("*.jsonl"))

meta_path = AGG_DIR / "run_metadata.json"
meta_path.write_text(json.dumps(RUN_META, ensure_ascii=False, indent=2), encoding="utf-8")
print("Metadata saved:", meta_path)
print(json.dumps({k: v for k, v in RUN_META.items() if k != "result_files"},
                 ensure_ascii=False, indent=2))

AttributeError: module 'datetime' has no attribute 'now'

---

## Appendix — Changes from the original notebooks

| Item | Original | This notebook | Reason |
|---|---|---|---|
| API keys | Hard-coded in the notebook | Environment variables / `.env` only | Prevent credential disclosure in submitted files |
| Model IDs | Mixed `gpt-4o` (alias) and `gpt-4o-2024-11-20` | Pinned snapshot IDs throughout | The alias resolves to a different snapshot, so methods ran on different models |
| LLM-Rank retry | Identical request repeated | Validation failure appended to the prompt | At `temperature = 0` an identical prompt cannot produce a different response |
| Error handling | `except` printed and dropped the result | Retry, then raise | Prevents silent shrinkage of per-run counts |
| Token usage | Not recorded | Recorded per call | Allows the reasoning-configuration asymmetry to be reported |
| Output paths | Mixed folder and naming conventions | `results/{method}/{model}/seed{n}.jsonl` | Simplifies aggregation |
| Vendor code | Copied across nine notebooks | One `chat()` adapter | Prevents prompt drift between vendors |
| `seed` | Passed for GPT only, undocumented | Same behaviour, documented | Anthropic and Google expose no seed parameter |

## Computation logic preserved

- **LLM-Score** — scores in [0, 1], five-decimal string format, averaged across repeats — unchanged
- **LLM-Seq** — greedy forward selection, `N - t + 1` points, averaged across repeats — unchanged
- **LMprior** — `top_logprobs = 20`, `FLOOR = -100`, `score = log P(Y) - log P(N)`,
  `y_observed` / `n_observed` flags — unchanged
- **LLM-Rank** — group split, per-group ranking, pairwise merge, averaged across repeats — unchanged